### Imports

In [4]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import numpy as np

In [ ]:
# Add project root to Python path so we can import trace package
project_root = Path().resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

# Verify trace package can be found
try:
    import trace

    print(f"✓ Found trace package at: {trace.__file__}")
except ImportError as e:
    print(f"✗ Error importing trace: {e}")
    print(f"  Current working directory: {Path.cwd()}")
    print(f"  Project root: {project_root}")
    print(f"  Python path: {sys.path[:3]}...")
    raise

from trace.statistics import compute_rd_pvalues
import pandas as pd

print("✓ Successfully imported compute_rd_pvalues")

### Data

In [3]:
cvd_binary_df = pd.read_csv("data/stats/cvd_stats_binary.txt")
cvd_numeric_df = pd.read_csv("data/stats/cvd_stats_numeric.txt")

diab_binary_df = pd.read_csv("data/stats/diab_stats_binary.txt")
diab_numeric_df = pd.read_csv("data/stats/diab_stats_numeric.txt")

plus50_binary_df = pd.read_csv("data/stats/plus50_stats_binary.txt")
plus50_numeric_df = pd.read_csv("data/stats/plus50_stats_numeric.txt")

In [ ]:
unit_dict = {
    "age_at_index_date": "years",
    "diabetes": "days",
    "estimated_glomerular_filtration_rate": "?",
    "ALT": "?",
    "hemoglobin_a1c": "mmol/mol?",
    "low_density_lipoprotein": "mmol/l?",
}

remove_rows = [
    "diabetes_icd10",
    "icd10_do24",
    "icd10_do244",
    "icd10_do24_excluding_do244",
    "diabetes_atc_antihyperglycemic",
]

### Functions

In [45]:
def prettify_name(name: str) -> str:
    """
    Custom renaming rules:
      - age_at_index -> Patient Age
      - diabetes -> Duration of diabetes
      - all others: replace '_' with ' ' and capitalize the first letter
    """
    if pd.isna(name):
        return ""

    if name == "age_at_index":
        return "Patient Age"
    if name == "diabetes":
        return "Duration of diabetes"

    name = name.replace("_", " ")
    return name[0].upper() + name[1:]


def format_cell(row: pd.Series, group: str):
    """
    Returns:
      formatted_value, type_flag ∈ {"mean_sd", "n_pct", "none"}
    """
    mean = row.get(f"mean_{group}", np.nan)
    sd = row.get(f"SD_{group}", np.nan)
    count = row.get(f"count_{group}", np.nan)
    pct = row.get(f"percentage_{group}", np.nan)

    if pd.notna(mean) and pd.notna(sd):
        return f"{mean:.2f} (±{sd:.2f})", "mean_sd"
    elif pd.notna(count) and pd.notna(pct):
        return f"{int(count):,} ({pct:.1f}%)", "n_pct"
    else:
        return np.nan, "none"


# ---------- Reshaping helpers ----------


def pivot_binary(binary_df: pd.DataFrame) -> pd.DataFrame:
    """Pivot and reshape binary_df into wide format."""
    if binary_df is None or binary_df.empty:
        return pd.DataFrame(columns=["criterion"])

    df_wide = binary_df.pivot(
        index="criterion", columns="group", values=["count_raw", "percentage_raw"]
    )

    df_wide.columns = [
        f"{'count' if m == 'count_raw' else 'percentage'}_{g}"
        for m, g in df_wide.columns
    ]

    return df_wide.reset_index()


def pivot_numeric(numeric_df: pd.DataFrame) -> pd.DataFrame:
    """
    Pivot numeric_df into wide format, after cleaning *_numeric_value.
    """
    if numeric_df is None or numeric_df.empty:
        return pd.DataFrame(columns=["criterion"])

    df_num = numeric_df.copy()

    df_num["criterion_clean"] = np.where(
        df_num["criterion"].str.endswith("_numeric_value", na=False),
        df_num["criterion"].str.replace(r"_numeric_value$", "", regex=True),
        df_num["criterion"],
    )

    stats_pivot = df_num.pivot(
        index="criterion_clean", columns="group", values=["mean_raw", "std_raw"]
    )

    stats_pivot.columns = [
        f"{'mean' if s == 'mean_raw' else 'SD'}_{g}" for s, g in stats_pivot.columns
    ]

    stats_pivot = stats_pivot.reset_index().rename(
        columns={"criterion_clean": "criterion"}
    )
    return stats_pivot


def merge_binary_numeric(binary_df, numeric_df) -> pd.DataFrame:
    """Merge wide binary and numeric tables."""
    wide_bin = pivot_binary(binary_df)
    wide_num = pivot_numeric(numeric_df)

    if wide_bin.empty and wide_num.empty:
        return pd.DataFrame()

    return pd.merge(wide_bin, wide_num, on="criterion", how="outer")


# ---------- Collapsing into final table ----------


def collapse_merged(
    merged_df: pd.DataFrame, groups=("Control", "Exposed"), units: dict | None = None
) -> pd.DataFrame:
    """
    Create the final formatted table.
    """
    if merged_df is None or merged_df.empty:
        return pd.DataFrame()

    rows = []

    for _, row in merged_df.iterrows():
        group_values = {}
        types = set()

        for g in groups:
            v, t = format_cell(row, g)
            group_values[g] = v
            types.add(t)

        base_name = row.get("criterion", np.nan)
        unit = None if units is None else units.get(base_name)

        # Suffix logic
        if "mean_sd" in types:
            suffix = f" (mean {unit}, SD)" if unit else " (mean, SD)"
        elif "n_pct" in types:
            suffix = " (n, %)"
        else:
            suffix = ""

        confounder_name = f"{prettify_name(base_name)}{suffix}"

        entry = {"Confounder": confounder_name}
        for g in groups:
            entry[g] = group_values[g]

        rows.append(entry)

    return pd.DataFrame(rows).set_index("Confounder")

### Make table

In [46]:
from pyarrow import binary


def build_confounder_table(
    binary_df: pd.DataFrame,
    numeric_df: pd.DataFrame,
    groups=("Control", "Exposed"),
    units: dict | None = None,
    remove_rows: list | None = None,
) -> pd.DataFrame:
    """
    High-level wrapper that:
      - removes unwanted criteria
      - reshapes binary and numeric data
      - merges them
      - formats rows and units
    """
    # Remove rows BEFORE pivoting
    if remove_rows:
        remove_rows_plus = [r + "_numeric_value" for r in remove_rows] + remove_rows
        binary_df = binary_df[~binary_df["criterion"].isin(remove_rows_plus)]
        numeric_df = numeric_df[~numeric_df["criterion"].isin(remove_rows_plus)]

    merged_df = merge_binary_numeric(binary_df, numeric_df)
    collapsed = collapse_merged(merged_df, groups=groups, units=units)
    return collapsed

In [47]:
cvd_collapsed = build_confounder_table(
    cvd_binary_df, cvd_numeric_df, units=unit_dict, remove_rows=remove_rows
)

diab_collapsed = build_confounder_table(
    diab_binary_df, diab_numeric_df, units=unit_dict, remove_rows=remove_rows
)

plus50_collapsed = build_confounder_table(
    plus50_binary_df, plus50_numeric_df, units=unit_dict, remove_rows=remove_rows
)

In [48]:
plus50_collapsed

,Control,Exposed
Confounder,,
"ALT (mean ?, SD)",27.84 (±20.87),34.28 (±21.82)
"ARB (n, %)","29,866 (12.0%)","2,364 (27.7%)"
"Ace inhibitors (n, %)","22,629 (9.1%)","1,884 (22.1%)"
"Age at index date (mean years, SD)",67.29 (±11.10),65.69 (±9.14)
"Antiplatelet therapy (n, %)","34,920 (14.1%)","2,736 (32.1%)"
"Beta blocks (n, %)","33,005 (13.3%)","2,477 (29.1%)"
"Chronic kidney disease (n, %)","2,644 (1.1%)",222 (2.6%)
"Depression (n, %)",0 (0.0%),0 (0.0%)
"Duration of diabetes (mean days, SD)",3242.98 (±2680.05),3894.94 (±2499.61)


### Combine tables

In [49]:
def combine_collapsed_tables(
    cvd_collapsed: pd.DataFrame,
    diab_collapsed: pd.DataFrame,
    plus50_collapsed: pd.DataFrame,
) -> pd.DataFrame:
    """
    Combine three collapsed tables side by side on the Confounder index.
    Each table has columns 'Control' and 'Exposed'.
    Result will have:
      Control_cvd, Exposed_cvd,
      Control_diab, Exposed_diab,
      Control_plus50, Exposed_plus50
    """
    # Make copies and add cohort-specific suffixes
    cvd = cvd_collapsed.rename(
        columns={"Control": "Control_cvd", "Exposed": "Exposed_cvd"}
    )
    diab = diab_collapsed.rename(
        columns={"Control": "Control_diab", "Exposed": "Exposed_diab"}
    )
    plus50 = plus50_collapsed.rename(
        columns={"Control": "Control_plus50", "Exposed": "Exposed_plus50"}
    )

    # Outer join on Confounder index
    combined = cvd.join(diab, how="outer").join(plus50, how="outer")

    # Reset index so Confounder becomes a column for Polars/great_tables
    combined = combined.reset_index().rename(columns={"Confounder": "confounder"})
    return combined


combined_df = combine_collapsed_tables(cvd_collapsed, diab_collapsed, plus50_collapsed)
combined_df

,confounder,Control_cvd,Exposed_cvd,Control_diab,Exposed_diab,Control_plus50,Exposed_plus50
0,"ALT (mean ?, SD)",28.33 (±22.91),34.94 (±24.12),28.66 (±21.57),35.28 (±23.13),27.84 (±20.87),34.28 (±21.82)
1,"ARB (n, %)","6,093 (11.9%)",527 (28.7%),"4,959 (21.6%)","2,429 (27.0%)","29,866 (12.0%)","2,364 (27.7%)"
2,"Ace inhibitors (n, %)","5,001 (9.8%)",430 (23.4%),"4,663 (20.3%)","1,986 (22.1%)","22,629 (9.1%)","1,884 (22.1%)"
3,"Age at index date (mean years, SD)",58.62 (±19.27),63.46 (±11.55),67.87 (±14.13),63.00 (±11.68),67.29 (±11.10),65.69 (±9.14)
4,"Antiplatelet therapy (n, %)","12,489 (24.4%)",856 (46.7%),"6,589 (28.6%)","2,761 (30.7%)","34,920 (14.1%)","2,736 (32.1%)"
5,"Beta blocks (n, %)","8,137 (15.9%)",646 (35.2%),"6,279 (27.3%)","2,471 (27.5%)","33,005 (13.3%)","2,477 (29.1%)"
6,"Chronic kidney disease (n, %)",805 (1.6%),64 (3.5%),863 (3.8%),222 (2.5%),"2,644 (1.1%)",222 (2.6%)
7,"Depression (n, %)",0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
8,"Diabetic nephropathy (n, %)",0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)
9,"Diabetic neuropathy (n, %)",0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%),0 (0.0%)


In [50]:
import polars as pl
import polars.selectors as cs

collapsed_pl = pl.from_pandas(combined_df)

In [51]:
from great_tables import loc, style

gt_tbl = (
    collapsed_pl.style
    # Title + subtitle for the paper
    .tab_header(
        title="Baseline characteristics by exposure status and cohort",
        subtitle="Values are mean (SD) or n (%) as indicated in the row label.",
    )
    # Use confounder as the stub (row names on the left)
    .tab_stub(rowname_col="confounder")
    # Column labels (per column)
    .cols_label(
        Control_cvd="Control",
        Exposed_cvd="Exposed",
        Control_diab="Control",
        Exposed_diab="Exposed",
        Control_plus50="Control",
        Exposed_plus50="Exposed",
    )
    # Column spanners over each pair of columns
    .tab_spanner("CVD cohort", ["Control_cvd", "Exposed_cvd"])
    .tab_spanner("Diabetes cohort", ["Control_diab", "Exposed_diab"])
    .tab_spanner("Age 50+ cohort", ["Control_plus50", "Exposed_plus50"])
    # Optional: make confounder names bold
    .tab_style(style.text(weight="bold"), loc.body(columns="confounder"))
    # Zebra-striping rows
    .opt_row_striping()
    # Outline around table
    .opt_table_outline()
    # Slightly more compact padding
    .opt_vertical_padding(scale=0.8)
    .opt_horizontal_padding(scale=1.0)
    # Serif font looks a bit more “paper-like”
    .opt_table_font(font="Times New Roman")
)

In [52]:
gt_tbl

GT(_tbl_data=shape: (25, 7)
┌──────────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────────┐
│ confounder   ┆ Control_cvd ┆ Exposed_cvd ┆ Control_dia ┆ Exposed_dia ┆ Control_plu ┆ Exposed_plu │
│ ---          ┆ ---         ┆ ---         ┆ b           ┆ b           ┆ s50         ┆ s50         │
│ str          ┆ str         ┆ str         ┆ ---         ┆ ---         ┆ ---         ┆ ---         │
│              ┆             ┆             ┆ str         ┆ str         ┆ str         ┆ str         │
╞══════════════╪═════════════╪═════════════╪═════════════╪═════════════╪═════════════╪═════════════╡
│ ALT (mean ?, ┆ 28.33       ┆ 34.94       ┆ 28.66       ┆ 35.28       ┆ 27.84       ┆ 34.28       │
│ SD)          ┆ (±22.91)    ┆ (±24.12)    ┆ (±21.57)    ┆ (±23.13)    ┆ (±20.87)    ┆ (±21.82)    │
│ ARB (n, %)   ┆ 6,093       ┆ 527 (28.7%) ┆ 4,959       ┆ 2,429       ┆ 29,866      ┆ 2,364       │
│              ┆ (11.9%)     ┆             ┆ (21.6%)     ┆ (27.0%)     ┆ (12.0%)     ┆ (27.7%)     │
│ Ace          ┆ 5,001       ┆ 430 (23.4%) ┆ 4,663       ┆ 1,986       ┆ 22,629      ┆ 1,884       │
│ inhibitors   ┆ (9.8%)      ┆             ┆ (20.3%)     ┆ (22.1%)     ┆ (9.1%)      ┆ (22.1%)     │
│ (n, %)       ┆             ┆             ┆             ┆             ┆             ┆             │
│ Age at index ┆ 58.62       ┆ 63.46       ┆ 67.87       ┆ 63.00       ┆ 67.29       ┆ 65.69       │
│ date (mean   ┆ (±19.27)    ┆ (±11.55)    ┆ (±14.13)    ┆ (±11.68)    ┆ (±11.10)    ┆ (±9.14)     │
│ years,…      ┆             ┆             ┆             ┆             ┆             ┆             │
│ Antiplatelet ┆ 12,489      ┆ 856 (46.7%) ┆ 6,589       ┆ 2,761       ┆ 34,920      ┆ 2,736       │
│ therapy (n,  ┆ (24.4%)     ┆             ┆ (28.6%)     ┆ (30.7%)     ┆ (14.1%)     ┆ (32.1%)     │
│ %)           ┆             ┆             ┆             ┆             ┆             ┆             │
│ …            ┆ …           ┆ …           ┆ …           ┆ …           ┆ …           ┆ …           │
│ Myocardial   ┆ 0 (0.0%)    ┆ 0 (0.0%)    ┆ 0 (0.0%)    ┆ 0 (0.0%)    ┆ 0 (0.0%)    ┆ 0 (0.0%)    │
│ infarction   ┆             ┆             ┆             ┆             ┆             ┆             │
│ (n, %)       ┆             ┆             ┆             ┆             ┆             ┆             │
│ Oral anticoa ┆ 4,997       ┆ 290 (15.8%) ┆ 3,362       ┆ 1,079       ┆ 21,389      ┆ 1,134       │
│ gulants (n,  ┆ (9.8%)      ┆             ┆ (14.6%)     ┆ (12.0%)     ┆ (8.6%)      ┆ (13.3%)     │
│ %)           ┆             ┆             ┆             ┆             ┆             ┆             │
│ Statins (n,  ┆ 15,141      ┆ 1,301       ┆ 12,543      ┆ 6,005       ┆ 55,982      ┆ 5,664       │
│ %)           ┆ (29.5%)     ┆ (70.9%)     ┆ (54.5%)     ┆ (66.8%)     ┆ (22.5%)     ┆ (66.4%)     │
│ Thiazides    ┆ 2,861       ┆ 215 (11.7%) ┆ 2,264       ┆ 1,086       ┆ 15,908      ┆ 1,090       │
│ (n, %)       ┆ (5.6%)      ┆             ┆ (9.8%)      ┆ (12.1%)     ┆ (6.4%)      ┆ (12.8%)     │
│ Type2        ┆ 4,487       ┆ 1,635       ┆ 22,957      ┆ 8,993       ┆ 20,285      ┆ 7,708       │
│ diabetes (n, ┆ (8.8%)      ┆ (89.1%)     ┆ (99.8%)     ┆ (100.0%)    ┆ (8.2%)      ┆ (90.4%)     │
│ %)           ┆             ┆             ┆             ┆             ┆             ┆             │
└──────────────┴─────────────┴─────────────┴─────────────┴─────────────┴─────────────┴─────────────┘, _body=<great_tables._gt_data.Body object at 0x3211b1a50>, _boxhead=Boxhead([ColInfo(var='confounder', type=<ColInfoTypeEnum.stub: 2>, column_label='confounder', column_align='left', column_width=None), ColInfo(var='Control_cvd', type=<ColInfoTypeEnum.default: 1>, column_label='Control', column_align='left', column_width=None), ColInfo(var='Exposed_cvd', type=<ColInfoTypeEnum.default: 1>, column_label='Exposed', column_align='left', column_width=None), ColInfo(var='Control_diab', type=<ColInfoTypeEnum.default: 1>, column_label='Cont

In [54]:
import os

os.makedirs("figures/table", exist_ok=True)
gt_tbl.save("figures/table/baseline_characteristics.png")

GT(_tbl_data=shape: (25, 7)
┌──────────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────────┬─────────────┐
│ confounder   ┆ Control_cvd ┆ Exposed_cvd ┆ Control_dia ┆ Exposed_dia ┆ Control_plu ┆ Exposed_plu │
│ ---          ┆ ---         ┆ ---         ┆ b           ┆ b           ┆ s50         ┆ s50         │
│ str          ┆ str         ┆ str         ┆ ---         ┆ ---         ┆ ---         ┆ ---         │
│              ┆             ┆             ┆ str         ┆ str         ┆ str         ┆ str         │
╞══════════════╪═════════════╪═════════════╪═════════════╪═════════════╪═════════════╪═════════════╡
│ ALT (mean ?, ┆ 28.33       ┆ 34.94       ┆ 28.66       ┆ 35.28       ┆ 27.84       ┆ 34.28       │
│ SD)          ┆ (±22.91)    ┆ (±24.12)    ┆ (±21.57)    ┆ (±23.13)    ┆ (±20.87)    ┆ (±21.82)    │
│ ARB (n, %)   ┆ 6,093       ┆ 527 (28.7%) ┆ 4,959       ┆ 2,429       ┆ 29,866      ┆ 2,364       │
│              ┆ (11.9%)     ┆             ┆ (21.6%)     ┆ (27.0%)     ┆ (12.0%)     ┆ (27.7%)     │
│ Ace          ┆ 5,001       ┆ 430 (23.4%) ┆ 4,663       ┆ 1,986       ┆ 22,629      ┆ 1,884       │
│ inhibitors   ┆ (9.8%)      ┆             ┆ (20.3%)     ┆ (22.1%)     ┆ (9.1%)      ┆ (22.1%)     │
│ (n, %)       ┆             ┆             ┆             ┆             ┆             ┆             │
│ Age at index ┆ 58.62       ┆ 63.46       ┆ 67.87       ┆ 63.00       ┆ 67.29       ┆ 65.69       │
│ date (mean   ┆ (±19.27)    ┆ (±11.55)    ┆ (±14.13)    ┆ (±11.68)    ┆ (±11.10)    ┆ (±9.14)     │
│ years,…      ┆             ┆             ┆             ┆             ┆             ┆             │
│ Antiplatelet ┆ 12,489      ┆ 856 (46.7%) ┆ 6,589       ┆ 2,761       ┆ 34,920      ┆ 2,736       │
│ therapy (n,  ┆ (24.4%)     ┆             ┆ (28.6%)     ┆ (30.7%)     ┆ (14.1%)     ┆ (32.1%)     │
│ %)           ┆             ┆             ┆             ┆             ┆             ┆             │
│ …            ┆ …           ┆ …           ┆ …           ┆ …           ┆ …           ┆ …           │
│ Myocardial   ┆ 0 (0.0%)    ┆ 0 (0.0%)    ┆ 0 (0.0%)    ┆ 0 (0.0%)    ┆ 0 (0.0%)    ┆ 0 (0.0%)    │
│ infarction   ┆             ┆             ┆             ┆             ┆             ┆             │
│ (n, %)       ┆             ┆             ┆             ┆             ┆             ┆             │
│ Oral anticoa ┆ 4,997       ┆ 290 (15.8%) ┆ 3,362       ┆ 1,079       ┆ 21,389      ┆ 1,134       │
│ gulants (n,  ┆ (9.8%)      ┆             ┆ (14.6%)     ┆ (12.0%)     ┆ (8.6%)      ┆ (13.3%)     │
│ %)           ┆             ┆             ┆             ┆             ┆             ┆             │
│ Statins (n,  ┆ 15,141      ┆ 1,301       ┆ 12,543      ┆ 6,005       ┆ 55,982      ┆ 5,664       │
│ %)           ┆ (29.5%)     ┆ (70.9%)     ┆ (54.5%)     ┆ (66.8%)     ┆ (22.5%)     ┆ (66.4%)     │
│ Thiazides    ┆ 2,861       ┆ 215 (11.7%) ┆ 2,264       ┆ 1,086       ┆ 15,908      ┆ 1,090       │
│ (n, %)       ┆ (5.6%)      ┆             ┆ (9.8%)      ┆ (12.1%)     ┆ (6.4%)      ┆ (12.8%)     │
│ Type2        ┆ 4,487       ┆ 1,635       ┆ 22,957      ┆ 8,993       ┆ 20,285      ┆ 7,708       │
│ diabetes (n, ┆ (8.8%)      ┆ (89.1%)     ┆ (99.8%)     ┆ (100.0%)    ┆ (8.2%)      ┆ (90.4%)     │
│ %)           ┆             ┆             ┆             ┆             ┆             ┆             │
└──────────────┴─────────────┴─────────────┴─────────────┴─────────────┴─────────────┴─────────────┘, _body=<great_tables._gt_data.Body object at 0x3211b1a50>, _boxhead=Boxhead([ColInfo(var='confounder', type=<ColInfoTypeEnum.stub: 2>, column_label='confounder', column_align='left', column_width=None), ColInfo(var='Control_cvd', type=<ColInfoTypeEnum.default: 1>, column_label='Control', column_align='left', column_width=None), ColInfo(var='Exposed_cvd', type=<ColInfoTypeEnum.default: 1>, column_label='Exposed', column_align='left', column_width=None), ColInfo(var='Control_diab', type=<ColInfoTypeEnum.default: 1>, column_label='Cont